## How to evaluate the performance of a Gen AI solution?

~ Model-centric or Technical Metrics - Easiest to optimize with

1. Loss (cross-entropy loss) 

2. Perplexity - how well your model is doing 

3. Accuracy 

4. Precision, Recall, F1, 

5. AUC-ROC

~ Business-centric or Outcome Metrics - Most tangible impact

1. KPIs tied to business objectives

2. ROI

3. Improvements in time, cost or resources

4. Customer satisfaction

5. Benchmark comparisons

# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td>
            <h2 style="color:#f71;">Reminder: OPTIONAL to execute C++ code or Rust code</h2>
            <span style="color:#f71;">As an alternative, you can run it on the website given yesterday</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use high end models GPT 5, Claude 4.5 Sonnet, Gemini 2.5 Pro, Grok 4, which are the slightly higher priced models. The costs are still low, but if you'd prefer to keep costs ultra low, please pick lower cost models like gpt-5-nano.
            </span>
        </td>
    </tr>
</table>

In [1]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display


In [2]:
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")



Google API Key exists and begins AI
Groq API Key exists and begins gsk_
OpenRouter API Key exists and begins sk-or-


In [3]:
# Connect to client libraries
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)



In [4]:
models = ["gemini-2.5-flash-lite", "llama3.2:latest", "qwen/qwen3-coder-free", "llama-3.3-70b-versatile", ]

clients = {"gemini-2.5-flash-lite": gemini, "llama-3.3-70b-versatile": groq, "llama3.2:latest": ollama, "qwen/qwen3-coder-free": openrouter}

# Want to keep costs ultra-low? Replace this with models of your choice, using the examples from yesterday

In [5]:
from system_info import retrieve_system_info, rust_toolchain_info

system_info = retrieve_system_info()
rust_info = rust_toolchain_info()
rust_info

{'installed': True,
 'rustc': {'path': 'C:\\Users\\KIIT\\.cargo\\bin\\rustc.EXE',
  'version': 'rustc 1.85.0 (4d91de4e4 2025-02-17)',
  'host_triple': 'x86_64-pc-windows-msvc',
  'release': '1.85.0',
  'commit_hash': '4d91de4e48198da2e33413efdcd9cd2cc0c46688'},
 'cargo': {'path': 'C:\\Users\\KIIT\\.cargo\\bin\\cargo.EXE',
  'version': 'cargo 1.85.0 (d73d2caf9 2024-12-31)'},
 'rustup': {'path': 'C:\\Users\\KIIT\\.cargo\\bin\\rustup.EXE',
  'version': 'rustup 1.28.1 (f9edccde0 2025-03-05)',
  'active_toolchain': 'stable-x86_64-pc-windows-msvc (default)',
  'default_toolchain': '',
  'toolchains': ['stable-x86_64-pc-windows-msvc (active, default)'],
  'targets_installed': ['x86_64-pc-windows-msvc']},
 'rust_analyzer': {'path': 'C:\\Users\\KIIT\\.cargo\\bin\\rust-analyzer.EXE'},
 'env': {'CARGO_HOME': 'C:\\Users\\KIIT\\.cargo',
  'RUSTUP_HOME': 'C:\\Users\\KIIT\\.rustup',
  'RUSTFLAGS': '',
  'CARGO_BUILD_TARGET': ''},
 'execution_examples': ['"C:\\Users\\KIIT\\.cargo\\bin\\cargo.EXE" buil

In [6]:
message = f"""
Here is a report of the system information for my computer.
I want to run a Rust compiler to compile a single rust file called main.rs and then execute it in the simplest way possible.
Please reply with whether I need to install a Rust toolchain to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile Rust code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.
Have the maximum possible runtime performance in mind; compile time can be slow. Fastest possible runtime performance for this platform is key.
Reply with the commands in markdown.

System information:
{system_info}

Rust toolchain information:
{rust_info}
"""

response = gemini.chat.completions.create(model=models[0], messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

You are already set up to compile and run Rust code.

For the fastest possible runtime performance on your platform, you should use the following `compile_command` and `run_command`:

**Compile Command:**

```bash
rustc main.rs --release --target x86_64-pc-windows-msvc -O -C opt-level=3 -C codegen-units=1 -C lto=fat
```

**Run Command:**

```bash
.\main.exe
```

**Explanation:**

*   **`rustc main.rs`**: This is the basic command to compile your `main.rs` file.
*   **`--release`**: This flag enables optimizations for release builds, which is crucial for runtime performance.
*   **`--target x86_64-pc-windows-msvc`**: While your system appears to be configured with MinGW, your `rustc` installation reports `x86_64-pc-windows-msvc` as its host triple. Explicitly specifying this target ensures that you are compiling for your native Windows environment using the MSVC toolchain, which generally yields better performance on Windows than MinGW for optimized builds. If you encounter issues with this target, you might need to install the MSVC build tools for Rust or ensure your system's C++ build tools are correctly configured.
*   **`-O`**: This is a shorthand for `-C opt-level=2`. It's good practice to include it explicitly or rely on `--release`.
*   **`-C opt-level=3`**: This flag instructs the compiler to apply the highest level of optimization. Level 3 enables more aggressive optimizations that can significantly improve runtime speed, although it might increase compile times.
*   **`-C codegen-units=1`**: This tells the compiler to perform LLVM optimizations on the entire crate as a single unit. This can lead to more pervasive optimizations but will increase compile times. For maximum runtime performance, this is generally preferred.
*   **`-C lto=fat`**: This enables Link-Time Optimization (LTO). LTO allows the compiler to perform optimizations across multiple compilation units (your entire program) during the linking phase. `fat` LTO provides the most aggressive cross-module optimizations, further boosting runtime performance at the cost of significantly longer compile times.

**Important Considerations:**

*   **C++ Build Tools (MSVC):** Your system information shows `gcc` and `g++` from MinGW, but your Rust toolchain reports `x86_64-pc-windows-msvc`. Rust on Windows often relies on the Microsoft C++ build tools (part of Visual Studio or Visual Studio Build Tools) for linking and system integration, even if you don't explicitly use `cl.exe`. If you encounter linking errors, you might need to install the "Desktop development with C++" workload from Visual Studio Installer.
*   **`main.exe`:** The compiled executable will be named `main.exe` by default when compiling `main.rs` with `rustc`.

This setup prioritizes runtime speed above all else, which aligns with your requirement.

## For C++, overwrite this with the commands from yesterday, or for Rust, use the new commands

Or just use the website like yesterday:

 https://www.programiz.com/cpp-programming/online-compiler/

In [7]:
compile_command = [
    "/Users/ed/.cargo/bin/rustc",
    "main.rs",
    "-C", "opt-level=3",
    "-C", "target-cpu=native",
    "-C", "codegen-units=1",
    "-C", "lto=fat",
    "-C", "panic=abort",
    "-C", "strip=symbols",
    "-o", "main",
]

run_command = ["./main"]


## And now, on with the main task

In [8]:
language = "Rust" # or "C++"
extension = "rs" if language == "Rust" else "cpp"

system_prompt = f"""
Your task is to convert Python code into high performance {language} code.
Respond only with {language} code. Do not provide any explanation other than occasional comments.
The {language} response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to {language} with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.{language} and then compiled and executed; the compilation command is:
{compile_command}
Respond only with {language} code.
Python code to port:

```python
{python}
```
"""

In [9]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [10]:
def write_output(code):
    with open(f"main.{extension}", "w") as f:
        f.write(code)

In [11]:
def port(model, python):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```rust','').replace('```','')
    return reply

In [12]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [16]:
def compile_and_run(code):
    import os
    import subprocess

    # Write generated code to file
    filename = f"main.{extension}"

    with open(filename, "w", encoding="utf-8") as f:
        f.write(code)

    # =========================
    # RUST
    # =========================
    if language == "Rust":

        compile_command = [
            "rustc",
            filename,
            "-C", "opt-level=3",
            "-C", "target-cpu=native",
            "-C", "codegen-units=1",
            "-C", "lto=fat",
            "-C", "panic=abort",
            "-o", "main"
        ]

    # =========================
    # C++
    # =========================
    else:
        compile_command = [
            "g++",
            filename,
            "-O3",
            "-march=native",
            "-std=c++17",
            "-o",
            "main"
        ]

    # Compile
    compile_process = subprocess.run(
        compile_command,
        text=True,
        capture_output=True
    )

    if compile_process.returncode != 0:
        return f"Compilation Error:\n\n{compile_process.stderr}"

    # Windows vs Linux/Mac
    executable = "main.exe" if os.name == "nt" else "./main"

    # Run executable
    run_process = subprocess.run(
        [executable],
        text=True,
        capture_output=True
    )

    if run_process.returncode != 0:
        return f"Runtime Error:\n\n{run_process.stderr}"

    return run_process.stdout

In [17]:
python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [22]:
from styles import CSS

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to {language}") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (original)",
                value=python_hard,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            cpp = gr.Code(
                label=f"{language} (generated)",
                value="",
                language="shell" if language == "Rust" else "cpp",
                lines=26
            )

    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        model = gr.Dropdown(models, value=models[0], show_label=False)
        convert = gr.Button(f"Port to {language}", elem_classes=["convert-btn"])
        cpp_run = gr.Button(f"Run {language}", elem_classes=["run-btn", "cpp"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Python result", lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
            cpp_out = gr.TextArea(label=f"{language} result", lines=8, elem_classes=["cpp-out"])

    convert.click(fn=port, inputs=[model, python], outputs=[cpp])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    cpp_run.click(fn=compile_and_run, inputs=[cpp], outputs=[cpp_out])

ui.launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
